# Module 01 — Build Canonical Interaction Matrix

Converts legacy phage–host interaction data from the main iGEM_Claremont_2026 repo into
the standardised format specified by INTERFACE.md §Module 01.

**Key decisions:**
- Positive pairs (`Affinity=1`): `source = 'literature_curated'`, `confidence = 0.8`
- Negative pairs (`Affinity=0`): `source` derived from `Source` field, `confidence = 0.5`
- Unknown/untested: `label = -1`
- phiL7 × Xcc ATCC 33913 known ground-truth pair added from Wang 2003 / Lee 2009.

---

# 模块 01 — 构建标准互动矩阵

将主 iGEM_Claremont_2026 仓库中的历史噬菌体–宿主互动数据转换为 INTERFACE.md §Module 01
规定的标准格式。

**主要决策：**
- 阳性配对（`Affinity=1`）：`source = 'literature_curated'`，`confidence = 0.8`
- 阴性配对（`Affinity=0`）：`source` 由 `Source` 字段推导，`confidence = 0.5`
- 未知/未测试：`label = -1`
- 从 Wang 2003 / Lee 2009 文献添加 phiL7 × Xcc ATCC 33913 已知基准配对。

## Cell 1: Library versions
## 第 1 格：库版本信息

In [1]:
import sys, subprocess, random
import pathlib, datetime

import pandas as pd
import numpy as np
import Bio

# Random seeds / 随机种子
random.seed(42)
np.random.seed(42)

print(f"Python  : {sys.version}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"biopython: {Bio.__version__}")

try:
    REPO_COMMIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], text=True
    ).strip()
except Exception:
    REPO_COMMIT_SHA = "unknown"
print(f"commit  : {REPO_COMMIT_SHA}")

Python  : 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]
pandas  : 2.3.3
numpy   : 2.3.4
biopython: 1.87
commit  : 05a41cd


## Cell 2: Path setup
## 第 2 格：路径设置

In [2]:
# Notebook lives at <REPO_ROOT>/01_data_ground_truth/processes/
# parents[1] from CWD = REPO_ROOT / 01_data_ground_truth
# parents[1].parent = REPO_ROOT
REPO_ROOT   = pathlib.Path.cwd().resolve().parents[1]
MOD01_DIR   = REPO_ROOT / "01_data_ground_truth"
OUTPUTS_DIR = MOD01_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Legacy data lives in the sibling iGEM_Claremont_2026 repo
# 历史数据位于同级的 iGEM_Claremont_2026 仓库中
LEGACY_REPO = REPO_ROOT.parent / "iGEM_Claremont_2026" / "01_data_ground_truth" / "outputs"

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"LEGACY_REPO : {LEGACY_REPO}")
print(f"Legacy exists: {LEGACY_REPO.exists()}")

REPO_ROOT   : /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth
LEGACY_REPO : /Users/alexy/Desktop/Claude Workspace/iGEM_Claremont_2026/01_data_ground_truth/outputs
Legacy exists: True


## Cell 3: Load legacy positive pairs
## 第 3 格：读取历史阳性配对

`phage_host_matrix_with_ids.csv` (long format) is the authoritative positive-pair source.
Columns: `Phage, Phage_Accession, Host_Name, Host_Accession, Affinity, Source`.

`phage_host_matrix_with_ids.csv`（长格式）是阳性配对的权威来源。
列：`Phage, Phage_Accession, Host_Name, Host_Accession, Affinity, Source`。

In [3]:
POSITIVES_PATH = LEGACY_REPO / "interaction_matrix" / "phage_host_matrix_with_ids.csv"
NEGATIVES_PATH = LEGACY_REPO / "negative_samples" / "negative_data_combined.csv"

if POSITIVES_PATH.exists():
    pos_df = pd.read_csv(POSITIVES_PATH, encoding="utf-8-sig")
    print(f"Loaded {len(pos_df)} positive pairs from {POSITIVES_PATH.name}")
    print(f"Columns: {list(pos_df.columns)}")
    print(pos_df.head(3).to_string(index=False))
else:
    print(f"WARN: {POSITIVES_PATH} not found — using empty DataFrame")
    pos_df = pd.DataFrame(columns=["Phage", "Phage_Accession", "Host_Name", "Host_Accession", "Affinity", "Source"])

if NEGATIVES_PATH.exists():
    neg_df = pd.read_csv(NEGATIVES_PATH, encoding="utf-8-sig")
    print(f"\nLoaded {len(neg_df)} negative pairs from {NEGATIVES_PATH.name}")
else:
    print(f"WARN: {NEGATIVES_PATH} not found — using empty DataFrame")
    neg_df = pd.DataFrame(columns=["Phage", "Phage_Accession", "Host_Name", "Host_Accession", "Affinity", "Source"])

Loaded 2235 positive pairs from phage_host_matrix_with_ids.csv
Columns: ['Phage', 'Phage_Accession', 'Host_Name', 'Host_Accession', 'Affinity', 'Source']
                                       Phage Phage_Accession                 Host_Name Host_Accession  Affinity           Source
Xanthomonas phage Carill600, complete genome      PX408739.1 Xanthomonas euvesicatoria    NZ_CP167225       1.0 NCBI: PX408739.1
  Xanthomonas phage Jill608, complete genome      PX408738.1 Xanthomonas euvesicatoria    NZ_CP167225       1.0 NCBI: PX408738.1
  Xanthomonas phage Juel629, complete genome      PX408737.1 Xanthomonas euvesicatoria    NZ_CP167225       1.0 NCBI: PX408737.1

Loaded 1901 negative pairs from negative_data_combined.csv


## Cell 4: Map to INTERFACE schema
## 第 4 格：映射到 INTERFACE 模式

Target schema (INTERFACE.md §Module 01):
`phage_acc, host_acc, host_organism, label, source, confidence, notes`

目标列名（INTERFACE.md §Module 01）：
`phage_acc, host_acc, host_organism, label, source, confidence, notes`

In [4]:
def map_source(raw_source: str, affinity: int) -> tuple:
    """Map legacy Source string to (source, confidence).
    将历史 Source 字段映射为 (source, confidence)。
    """
    s = str(raw_source).lower()
    if affinity == 1:
        # Positive pairs came from literature / 阳性配对来自文献
        return "literature_curated", 0.8
    elif "module_a" in s or "cross_genus" in s:
        # Cross-genus inference / 跨属推断陰性
        return "inferred_taxonomy", 0.5
    elif "module_c" in s or "pv_inference" in s:
        # Pathovar-inference negatives / 致病型推断陰性
        return "inferred_taxonomy", 0.5
    else:
        return "literature_curated", 0.5


def build_rows(df: pd.DataFrame, default_label: int) -> list:
    """Convert legacy long-format rows to INTERFACE-schema dicts.
    将历史长格式行转换为 INTERFACE 模式字典列表。
    """
    rows = []
    for _, r in df.iterrows():
        phage_acc = str(r.get("Phage_Accession", "")).strip()
        host_acc  = str(r.get("Host_Accession",  "")).strip()
        host_org  = str(r.get("Host_Name",       "")).strip()
        affinity  = r.get("Affinity", default_label)
        raw_src   = r.get("Source", "")

        # Skip rows with missing accessions or placeholder text
        # 跳过登录号缺失或含占位符文本的行
        if not phage_acc or phage_acc in ("nan", ""):
            continue
        if host_acc in ("No Complete Genome Found", "nan", ""):
            host_acc = ""  # keep row but mark accession missing

        try:
            label = int(float(affinity))
        except (ValueError, TypeError):
            label = -1

        if label not in (0, 1, -1):
            label = -1

        src, conf = map_source(raw_src, label)
        rows.append({
            "phage_acc":    phage_acc,
            "host_acc":     host_acc,
            "host_organism": host_org,
            "label":        label,
            "source":       src,
            "confidence":   round(conf, 6),
            "notes":        str(raw_src)[:200],
        })
    return rows


pos_rows = build_rows(pos_df, default_label=1)
neg_rows = build_rows(neg_df, default_label=0)
print(f"Positive pairs mapped : {len(pos_rows)}")
print(f"Negative pairs mapped : {len(neg_rows)}")

Positive pairs mapped : 2235
Negative pairs mapped : 1901


## Cell 5: Add phiL7 × Xcc ATCC 33913 ground-truth row
## 第 5 格：添加 phiL7 × Xcc ATCC 33913 基准行

Wang et al. 2003 (*Mol Microbiol* 48:1049) identified TonB-ExbB-ExbD1D2 as phiL7 receptor.
Lee et al. 2009 (*AEM* 75:7828) characterised phiL7 genome (EU717894.1) and confirmed
host is *Xanthomonas campestris* pv. *campestris* ATCC 33913 (GCF_000007145.1).

This row is the primary positive training signal for Module 06.

Wang et al. 2003 确认 TonB-ExbB-ExbD1D2 为 phiL7 受体；Lee et al. 2009 鉴定了
phiL7 基因组（EU717894.1），并确认宿主为黄单胞菌 pv. campestris ATCC 33913（GCF_000007145.1）。

此行是 Module 06 的主要阳性训练信号。

In [5]:
# High-confidence ground-truth row for the reference scaffold
# 参考支架的高置信度基准行
GROUND_TRUTH = {
    "phage_acc":     "EU717894.1",
    "host_acc":      "GCF_000007145.1",
    "host_organism": "Xanthomonas campestris pv. campestris ATCC 33913",
    "label":         1,
    "source":        "literature_curated",
    "confidence":    0.95,
    "notes":         "Wang 2003 Mol Microbiol 48:1049 (receptor); Lee 2009 AEM 75:7828 (genome)",
}
print("Ground-truth row:")
for k, v in GROUND_TRUTH.items():
    print(f"  {k}: {v}")

Ground-truth row:
  phage_acc: EU717894.1
  host_acc: GCF_000007145.1
  host_organism: Xanthomonas campestris pv. campestris ATCC 33913
  label: 1
  source: literature_curated
  confidence: 0.95
  notes: Wang 2003 Mol Microbiol 48:1049 (receptor); Lee 2009 AEM 75:7828 (genome)


## Cell 6: Merge, deduplicate, validate
## 第 6 格：合并、去重、验证

In [6]:
all_rows = pos_rows + neg_rows + [GROUND_TRUTH]
matrix = pd.DataFrame(all_rows)

print(f"Before dedup: {len(matrix)} rows")

# Deduplicate on (phage_acc, host_acc) keeping highest-confidence row
# 按 (phage_acc, host_acc) 去重，保留置信度最高的行
matrix = (
    matrix
    .sort_values("confidence", ascending=False)
    .drop_duplicates(subset=["phage_acc", "host_acc"], keep="first")
    .reset_index(drop=True)
)
print(f"After dedup : {len(matrix)} rows")

# Validate schema / 验证列名
REQUIRED_COLS = ["phage_acc", "host_acc", "host_organism", "label", "source", "confidence", "notes"]
assert set(REQUIRED_COLS) <= set(matrix.columns), f"Missing cols: {set(REQUIRED_COLS) - set(matrix.columns)}"

# Validate label values / 验证 label 取值
assert matrix["label"].isin([1, 0, -1]).all(), "label must be 1, 0, or -1"

# Validate confidence range / 验证 confidence 范围
assert matrix["confidence"].between(0.0, 1.0).all(), "confidence must be in [0,1]"

# Replace NaN with empty string per CSV convention / 按 CSV 约定将 NaN 替换为空字符串
matrix = matrix.fillna("")

print("\nLabel distribution / 标签分布:")
print(matrix["label"].value_counts().to_string())
print("\nSource distribution / 来源分布:")
print(matrix["source"].value_counts().to_string())
print(f"\nPhage accessions: {matrix['phage_acc'].nunique()} unique")
print(f"Host accessions : {matrix['host_acc'].nunique()} unique")

Before dedup: 4137 rows
After dedup : 2236 rows

Label distribution / 标签分布:
label
 0    1901
 1     317
-1      18

Source distribution / 来源分布:
source
inferred_taxonomy     1901
literature_curated     335

Phage accessions: 777 unique
Host accessions : 38 unique


## Cell 7: Write interaction_matrix.csv
## 第 7 格：写出 interaction_matrix.csv

In [7]:
out_path = OUTPUTS_DIR / "interaction_matrix.csv"
matrix[REQUIRED_COLS].to_csv(out_path, index=False, float_format="%.6f")
print(f"Wrote {len(matrix)} rows to {out_path}")
print(matrix[REQUIRED_COLS].head(5).to_string(index=False))

Wrote 2236 rows to /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth/01_data_ground_truth/outputs/interaction_matrix.csv


  phage_acc        host_acc                                    host_organism  label             source  confidence                                                                     notes
 EU717894.1 GCF_000007145.1 Xanthomonas campestris pv. campestris ATCC 33913      1 literature_curated        0.95 Wang 2003 Mol Microbiol 48:1049 (receptor); Lee 2009 AEM 75:7828 (genome)
NC_054459.1     NZ_CP150073                    Xanthomonas oryzae pv. oryzae      1 literature_curated        0.80                                                         NCBI: NC_054459.1
 ON758385.1                                                  Xanthomonas sp.      1 literature_curated        0.80                                                          NCBI: ON758385.1
 ON711490.1                                     Xanthomonas campestris XC114      1 literature_curated        0.80                                                          NCBI: ON711490.1
 OP067662.1     NZ_CP150073                    Xanthom

## Cell 8: Update outputs/MANIFEST.csv with interaction_matrix.csv entry
## 第 8 格：将 interaction_matrix.csv 条目写入 outputs/MANIFEST.csv

In [8]:
import hashlib, csv

def sha256_file(path: pathlib.Path) -> str:
    """SHA-256 hex digest. / SHA-256 十六进制摘要。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def append_manifest_row(manifest_path: pathlib.Path, row: dict) -> None:
    """Append one row to MANIFEST.csv. / 向 MANIFEST.csv 追加一行。"""
    fieldnames = ["filename", "sha256", "bytes", "n_records",
                  "created_utc", "source_acc", "source_module", "notes"]
    write_header = not manifest_path.exists()
    with open(manifest_path, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        if write_header:
            w.writeheader()
        w.writerow(row)

manifest_path = OUTPUTS_DIR / "MANIFEST.csv"
NOW_UTC = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Remove old interaction_matrix entry if present / 移除旧的 interaction_matrix 条目
if manifest_path.exists():
    mdf = pd.read_csv(manifest_path)
    mdf = mdf[mdf["filename"] != "interaction_matrix.csv"]
    mdf.to_csv(manifest_path, index=False)

append_manifest_row(manifest_path, {
    "filename":      "interaction_matrix.csv",
    "sha256":        sha256_file(out_path),
    "bytes":         out_path.stat().st_size,
    "n_records":     len(matrix),
    "created_utc":   NOW_UTC,
    "source_acc":    "",
    "source_module": "01_data_ground_truth",
    "notes":         f"phage-host interaction matrix: {len(matrix)} pairs",
})
print(f"Updated MANIFEST.csv — {len(pd.read_csv(manifest_path))} total rows")

Updated MANIFEST.csv — 10 total rows


## Cell 9: Verification — phiL7 and key stats
## 第 9 格：验证 — phiL7 及关键统计

Assert that the canonical phiL7 × Xcc row is present with high confidence.

断言标准 phiL7 × Xcc 行存在且置信度 ≥ 0.9。

In [9]:
mat = pd.read_csv(out_path)

# Assert phiL7 row exists / 断言 phiL7 行存在
phil7_rows = mat[mat["phage_acc"].str.contains("EU717894", na=False)]
assert len(phil7_rows) >= 1, "phiL7 (EU717894.1) not in interaction_matrix.csv!"
print(f"phiL7 rows found: {len(phil7_rows)}")
print(phil7_rows.to_string(index=False))

# Assert ground-truth row / 断言基准行存在
gt = mat[
    mat["phage_acc"].str.contains("EU717894", na=False) &
    mat["host_acc"].str.contains("GCF_000007145", na=False)
]
assert len(gt) == 1, "Ground-truth phiL7 × Xcc row missing!"
assert gt.iloc[0]["label"] == 1, "Ground-truth label should be 1 (lyses)"
assert gt.iloc[0]["confidence"] >= 0.9, "Ground-truth confidence should be ≥ 0.9"
print("\n✓ Ground-truth phiL7 × Xcc row verified.")

# Summary stats / 汇总统计
print(f"\nTotal rows : {len(mat)}")
print(f"Labels     : {dict(mat['label'].value_counts())}")
print(f"Sources    : {dict(mat['source'].value_counts())}")
print("\n✓ Interaction matrix build complete. / 互动矩阵构建完成。")

phiL7 rows found: 3
 phage_acc        host_acc                                    host_organism  label             source  confidence                                                                     notes
EU717894.1 GCF_000007145.1 Xanthomonas campestris pv. campestris ATCC 33913      1 literature_curated        0.95 Wang 2003 Mol Microbiol 48:1049 (receptor); Lee 2009 AEM 75:7828 (genome)
EU717894.1     NZ_CP155977            Xanthomonas campestris pv. campestris      1 literature_curated        0.80                                                          NCBI: EU717894.1
EU717894.1     NZ_CP167225           Xanthomonas campestris pv. vesicatoria      0  inferred_taxonomy        0.50 Module_C_pv_inference|positive_host:Xanthomonas campestris pv. campestris

✓ Ground-truth phiL7 × Xcc row verified.

Total rows : 2236
Labels     : {0: np.int64(1901), 1: np.int64(317), -1: np.int64(18)}
Sources    : {'inferred_taxonomy': np.int64(1901), 'literature_curated': np.int64(335)}

✓ Interac